In [1]:
!pip install -q albumentations segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.7 MB/s eta 0:00:00


In [2]:
import os
import random
import json
import time
from dataclasses import dataclass

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
@dataclass
class CFG:
    root_path: str = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train"
    image_dir: str = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/images"
    mask_dir: str = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/masks"
    save_dir: str = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/output_deeplabv3plus"

    seed: int = 42
    image_size: int = 256
    in_channels: int = 1
    num_classes: int = 2
    val_size: float = 0.2

    batch_size: int = 8
    num_workers: int = 2
    epochs: int = 60
    lr: float = 1e-3
    weight_decay: float = 1e-4
    min_lr: float = 1e-6

    encoder_name: str = "resnet34"
    fourier_modes: int = 8
    fourier_hidden_channels: int = 16

    monitor_metric: str = "dice"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp: bool = torch.cuda.is_available()

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)

print(cfg)


CFG(root_path='/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train', image_dir='/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/images', mask_dir='/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/masks', save_dir='/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/output_deeplabv3plus', seed=42, image_size=256, in_channels=1, num_classes=2, val_size=0.2, batch_size=8, num_workers=2, epochs=60, lr=0.001, weight_decay=0.0001, min_lr=1e-06, encoder_name='resnet34', fourier_modes=8, fourier_hidden_channels=16, monitor_metric='dice', device='cpu', use_amp=False)


In [4]:
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

In [5]:
def build_file_pairs(image_dir, mask_dir):
    image_files = sorted(os.listdir(image_dir))
    pairs = []

    for fname in image_files:
        image_path = os.path.join(image_dir, fname)
        mask_path = os.path.join(mask_dir, fname)

        if os.path.isfile(image_path) and os.path.isfile(mask_path):
            pairs.append((image_path, mask_path))

    if len(pairs) == 0:
        raise RuntimeError("Nem találtam image-mask párokat. Ellenőrizd a mappaneveket.")

    return pairs

In [6]:
def create_train_val_split(pairs, val_size=0.2, seed=42):
    train_pairs, val_pairs = train_test_split(
        pairs,
        test_size=val_size,
        random_state=seed,
        shuffle=True
    )
    return train_pairs, val_pairs

In [7]:
def get_transforms(train=True, image_size=256):
    mean = (0.5,)
    std = (0.5,)

    if train:
        return A.Compose([
            A.Resize(image_size, image_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=15,
                border_mode=0,
                p=0.5
            ),
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(mean=mean, std=std, max_pixel_value=255.0),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(image_size, image_size),
            A.Normalize(mean=mean, std=std, max_pixel_value=255.0),
            ToTensorV2()
        ])

In [8]:
class BrainMRISegDataset(Dataset):
    def __init__(self, pairs, transforms=None, num_classes=2):
        self.pairs = pairs
        self.transforms = transforms
        self.num_classes = num_classes

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = np.array(Image.open(image_path).convert("L"))
        mask = np.array(Image.open(mask_path))

        if self.num_classes == 2:
            if mask.max() > 1:
                mask = (mask > 0).astype(np.uint8)
        else:
            mask = mask.astype(np.uint8)

        if self.transforms is not None:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        mask = mask.long()

        return image, mask

In [11]:
import segmentation_models_pytorch as smp


@torch.jit.script
def compl_mul2d_fno(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Complex multiplication for 2D tensors in Fourier space."""
    return torch.einsum("bixy,ioxy->boxy", a, b)


class SpectralConv2d_fno(nn.Module):
    """Spectral convolution using FFT for Fourier Neural Operator."""

    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2

        self.scale = 1 / max(1, in_channels * out_channels)
        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )

    def forward(self, x):
        batchsize = x.shape[0]
        height, width = x.size(-2), x.size(-1)

        modes1 = min(self.modes1, height)
        modes2 = min(self.modes2, width // 2 + 1)

        x_ft = torch.fft.rfftn(x, dim=[2, 3])
        out_ft = torch.zeros(
            batchsize,
            self.out_channels,
            height,
            width // 2 + 1,
            device=x.device,
            dtype=torch.cfloat,
        )

        out_ft[:, :, :modes1, :modes2] = compl_mul2d_fno(
            x_ft[:, :, :modes1, :modes2], self.weights1[:, :, :modes1, :modes2]
        )
        out_ft[:, :, -modes1:, :modes2] = compl_mul2d_fno(
            x_ft[:, :, -modes1:, :modes2], self.weights2[:, :, :modes1, :modes2]
        )

        x = torch.fft.irfftn(out_ft, s=(height, width), dim=[2, 3])
        return x


class FourierLayer_fno(nn.Module):
    """Spatial conv + spectral conv."""

    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, modes=4):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=True,
        )
        self.conv_fno = SpectralConv2d_fno(in_channels, out_channels, modes, modes)

    def forward(self, x):
        return self.conv(x) + self.conv_fno(x)


class DeepLabV3PlusFourier(nn.Module):



    def __init__(
        self,
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=1,
        classes=2,
        fourier_hidden_channels=16,
        fourier_modes=8,
    ):
        super().__init__()

        self.fourier_stem = nn.Sequential(
            FourierLayer_fno(
                in_channels=in_channels,
                out_channels=fourier_hidden_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                modes=fourier_modes,
            ),
            nn.BatchNorm2d(fourier_hidden_channels),
            nn.ReLU(inplace=True),
            FourierLayer_fno(
                in_channels=fourier_hidden_channels,
                out_channels=fourier_hidden_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                modes=fourier_modes,
            ),
            nn.BatchNorm2d(fourier_hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(fourier_hidden_channels, in_channels, kernel_size=1),
        )

        self.deeplab = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=classes,
        )

    def forward(self, x):
        x = x + self.fourier_stem(x)
        return self.deeplab(x)


model = DeepLabV3PlusFourier(
    encoder_name=cfg.encoder_name,
    encoder_weights=None,
    in_channels=cfg.in_channels,
    classes=cfg.num_classes,
    fourier_hidden_channels=cfg.fourier_hidden_channels,
    fourier_modes=cfg.fourier_modes,
)

model = model.to(cfg.device)
print("Model ready on:", cfg.device)
print(f"Fourier + DeepLabV3+ | encoder={cfg.encoder_name} | modes={cfg.fourier_modes}")


Model ready on: cpu
Fourier + DeepLabV3+ | encoder=resnet34 | modes=8


In [12]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6, num_classes=2):
        super().__init__()
        self.smooth = smooth
        self.num_classes = num_classes

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dims)
        union = torch.sum(probs, dims) + torch.sum(targets_one_hot, dims)

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)

        if self.num_classes > 1:
            dice = dice[1:]

        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss(num_classes=num_classes)

    def forward(self, logits, targets):
        return 0.5 * self.ce(logits, targets) + 0.5 * self.dice(logits, targets)

In [13]:
@torch.no_grad()
def compute_dice_iou(logits, targets, num_classes=2, eps=1e-6):
    preds = torch.argmax(logits, dim=1)

    dices = []
    ious = []

    for cls in range(1, num_classes):
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        dice = (2 * intersection + eps) / (union + eps)

        union_iou = pred_cls.sum() + target_cls.sum() - intersection
        iou = (intersection + eps) / (union_iou + eps)

        dices.append(dice.item())
        ious.append(iou.item())

    return float(np.mean(dices)), float(np.mean(ious))

In [14]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device).float()
        masks = masks.to(device).long()

        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()

        dice, iou = compute_dice_iou(logits.detach(), masks, num_classes=cfg.num_classes)

        total_loss += loss.item()
        total_dice += dice
        total_iou += iou

    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

In [15]:
@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device).float()
        masks = masks.to(device).long()

        logits = model(images)
        loss = criterion(logits, masks)

        dice, iou = compute_dice_iou(logits, masks, num_classes=cfg.num_classes)

        total_loss += loss.item()
        total_dice += dice
        total_iou += iou

    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

In [16]:
import os

root_path = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train"

for root, dirs, files in os.walk(root_path):
    print("ROOT:", root)
    print("DIRS:", dirs[:10])
    print("FILES:", files[:10])
    print("-" * 80)

ROOT: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train
DIRS: ['images', 'masks']
FILES: []
--------------------------------------------------------------------------------
ROOT: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/images
DIRS: []
FILES: ['brisc2025_train_04011_pi_co_t1.jpg', 'brisc2025_train_04000_pi_co_t1.jpg', 'brisc2025_train_03999_pi_co_t1.jpg', 'brisc2025_train_03980_pi_co_t1.jpg', 'brisc2025_train_04008_pi_co_t1.jpg', 'brisc2025_train_04015_pi_co_t1.jpg', 'brisc2025_train_04018_pi_co_t1.jpg', 'brisc2025_train_04012_pi_co_t1.jpg', 'brisc2025_train_04010_pi_co_t1.jpg', 'brisc2025_train_03965_pi_ax_t1.jpg']
--------------------------------------------------------------------------------
ROOT: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/masks
DIRS: []
FILES: ['brisc2025_train_03983_pi_co_t1.png', 'brisc2025_train_03980_pi_co_t1.png', 'brisc2025_train_04002_pi_co_t1.png', 'brisc2025_train_03998_pi_co_t1.png', 'bri

In [17]:
def build_file_pairs(image_dir, mask_dir):
    image_files = [
        f for f in os.listdir(image_dir)
        if os.path.isfile(os.path.join(image_dir, f))
    ]

    mask_files = [
        f for f in os.listdir(mask_dir)
        if os.path.isfile(os.path.join(mask_dir, f))
    ]

    # basename -> teljes elérési út
    image_dict = {}
    for f in image_files:
        name = os.path.splitext(f)[0]
        image_dict[name] = os.path.join(image_dir, f)

    mask_dict = {}
    for f in mask_files:
        name = os.path.splitext(f)[0]
        mask_dict[name] = os.path.join(mask_dir, f)

    # közös kulcsok
    common_keys = sorted(set(image_dict.keys()) & set(mask_dict.keys()))

    print("Összes kép:", len(image_dict))
    print("Összes maszk:", len(mask_dict))
    print("Közös párok:", len(common_keys))

    if len(common_keys) == 0:
        raise RuntimeError("Nincs egyező fájlnév a képek és maszkok között.")

    pairs = [(image_dict[k], mask_dict[k]) for k in common_keys]

    return pairs

In [18]:
criterion = CombinedLoss(num_classes=cfg.num_classes)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=cfg.min_lr
)

scaler = torch.cuda.amp.GradScaler() if cfg.use_amp else None

In [19]:
best_dice = -1.0
history = {
    "train_loss": [],
    "train_dice": [],
    "train_iou": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": []
}

best_model_path = os.path.join(cfg.save_dir, "best_deeplabv3plus_fourier_brain_mri.pth")

In [20]:
pairs = build_file_pairs(cfg.image_dir, cfg.mask_dir)
train_pairs, val_pairs = create_train_val_split(pairs, cfg.val_size, cfg.seed)

train_ds = BrainMRISegDataset(
    train_pairs,
    transforms=get_transforms(train=True, image_size=cfg.image_size),
    num_classes=cfg.num_classes
)

val_ds = BrainMRISegDataset(
    val_pairs,
    transforms=get_transforms(train=False, image_size=cfg.image_size),
    num_classes=cfg.num_classes
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

Összes kép: 3933
Összes maszk: 3933
Közös párok: 3933


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
for epoch in range(cfg.epochs):
    start_time = time.time()

    train_loss, train_dice, train_iou = train_one_epoch(
        model, train_loader, optimizer, criterion, cfg.device, scaler
    )

    val_loss, val_dice, val_iou = valid_one_epoch(
        model, val_loader, criterion, cfg.device
    )

    scheduler.step(val_dice)

    history["train_loss"].append(train_loss)
    history["train_dice"].append(train_dice)
    history["train_iou"].append(train_iou)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{cfg.epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f} | Train IoU: {train_iou:.4f} || "
        f"Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f} || "
        f"Time: {elapsed:.1f}s"
    )

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_dice": best_dice,
            "config": cfg.__dict__
        }, best_model_path)

        print(f"--> Új legjobb modell mentve. Val Dice: {best_dice:.6f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/60] Train Loss: 0.2881 | Train Dice: 0.5261 | Train IoU: 0.3739 || Val Loss: 0.2232 | Val Dice: 0.6304 | Val IoU: 0.4743 || Time: 1404.1s
--> Új legjobb modell mentve. Val Dice: 0.630440


In [ ]:
history_path = os.path.join(cfg.save_dir, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print("Training kész.")
print("Best Dice:", best_dice)
print("Model mentve ide:", best_model_path)

In [ ]:
history_path = os.path.join(cfg.save_dir, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print("Training kész.")
print("Best Dice:", best_dice)
print("Model mentve ide:", best_model_path)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

In [ ]:
def plot_training_history(history, save_dir):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["train_dice"], label="Train Dice")
    plt.plot(epochs, history["val_dice"], label="Val Dice")
    plt.title("Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["train_iou"], label="Train IoU")
    plt.plot(epochs, history["val_iou"], label="Val IoU")
    plt.title("IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    save_path = os.path.join(save_dir, "training_curves.png")
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Training curves mentve:", save_path)

In [ ]:
plot_training_history(history, cfg.save_dir)

In [ ]:
checkpoint = torch.load(best_model_path, map_location=cfg.device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best model loaded from:", best_model_path)

In [ ]:
@torch.no_grad()
def predict_mask(model, image_tensor, device):
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device).float()   # [1, C, H, W]
    logits = model(image_tensor)
    pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    return pred

In [ ]:
def denormalize_grayscale(img_tensor):
    # img_tensor: [1, H, W]
    img = img_tensor.squeeze(0).cpu().numpy()
    img = img * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    return img

In [ ]:
def make_overlay(image_gray, pred_mask, alpha=0.35):
    """
    image_gray: [H, W] in [0,1]
    pred_mask: [H, W] {0,1}
    """
    image_rgb = np.stack([image_gray, image_gray, image_gray], axis=-1)

    overlay = image_rgb.copy()
    red_mask = np.zeros_like(image_rgb)
    red_mask[..., 0] = 1.0

    mask_region = pred_mask > 0
    overlay[mask_region] = (1 - alpha) * overlay[mask_region] + alpha * red_mask[mask_region]
    return overlay

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=4, save_dir=None):
    indices = np.linspace(0, len(dataset) - 1, num_samples, dtype=int)

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))

    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        image_tensor, mask_tensor = dataset[idx]

        pred_mask = predict_mask(model, image_tensor, device)
        image_gray = denormalize_grayscale(image_tensor)
        gt_mask = mask_tensor.cpu().numpy()
        overlay = make_overlay(image_gray, pred_mask)

        axes[row, 0].imshow(image_gray, cmap="gray")
        axes[row, 0].set_title("Original Image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt_mask, cmap="gray")
        axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_mask, cmap="gray")
        axes[row, 2].set_title("Predicted Mask")
        axes[row, 2].axis("off")

        axes[row, 3].imshow(overlay)
        axes[row, 3].set_title("Overlay")
        axes[row, 3].axis("off")

    plt.tight_layout()

    if save_dir is not None:
        save_path = os.path.join(save_dir, "prediction_examples.png")
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Prediction figure mentve:", save_path)

    plt.show()

In [ ]:
visualize_predictions(
    model=model,
    dataset=val_ds,
    device=cfg.device,
    num_samples=4,
    save_dir=cfg.save_dir
)

In [ ]:
@torch.no_grad()
def compute_sample_iou(pred_mask, true_mask, eps=1e-6):
    pred = (pred_mask > 0).astype(np.uint8)
    true = (true_mask > 0).astype(np.uint8)

    intersection = np.logical_and(pred, true).sum()
    union = np.logical_or(pred, true).sum()

    return (intersection + eps) / (union + eps)

In [ ]:
def infer_tumor_type_from_path(image_path: str):
    p = image_path.lower()

    if "glioma" in p:
        return "Glioma"
    elif "meningioma" in p:
        return "Meningioma"
    elif "pituitary" in p or "hypophysis" in p:
        return "Pituitary"
    else:
        return "Unknown"

In [ ]:
@torch.no_grad()
def evaluate_by_tumor_type(model, dataset, device, model_name="DeepLabV3+"):
    per_type_ious = {
        "Glioma": [],
        "Meningioma": [],
        "Pituitary": [],
        "Unknown": []
    }

    for idx in range(len(dataset)):
        image_tensor, mask_tensor = dataset[idx]
        pred_mask = predict_mask(model, image_tensor, device)
        true_mask = mask_tensor.cpu().numpy()

        image_path, _ = dataset.pairs[idx]
        tumor_type = infer_tumor_type_from_path(image_path)

        iou = compute_sample_iou(pred_mask, true_mask)
        per_type_ious[tumor_type].append(iou)

    def safe_mean(values):
        return float(np.mean(values)) if len(values) > 0 else np.nan

    glioma_miou = safe_mean(per_type_ious["Glioma"])
    meningioma_miou = safe_mean(per_type_ious["Meningioma"])
    pituitary_miou = safe_mean(per_type_ious["Pituitary"])

    known_values = (
        per_type_ious["Glioma"] +
        per_type_ious["Meningioma"] +
        per_type_ious["Pituitary"]
    )
    weighted_miou = safe_mean(known_values)

    df = pd.DataFrame([{
        "Model": model_name,
        "mIoU Glioma": glioma_miou,
        "mIoU Meningioma": meningioma_miou,
        "mIoU Pituitary": pituitary_miou,
        "Weighted mIoU": weighted_miou
    }])

    return df, per_type_ious

In [ ]:
results_df, per_type_ious = evaluate_by_tumor_type(
    model=model,
    dataset=val_ds,
    device=cfg.device,
    model_name="DeepLabV3+"
)

print(results_df)

In [ ]:
def display_results_table(df, save_dir=None):
    df_rounded = df.copy()

    for col in df_rounded.columns:
        if col != "Model":
            df_rounded[col] = df_rounded[col].map(lambda x: round(x, 4) if pd.notnull(x) else x)

    print(df_rounded)

    if save_dir is not None:
        csv_path = os.path.join(save_dir, "miou_results.csv")
        df_rounded.to_csv(csv_path, index=False)
        print("Táblázat mentve:", csv_path)

    return df_rounded